# Friend Project — GPU-Fuzzy Pipeline (Per-Symbol Phase 2)

This notebook runs the friend's GPU-Fuzzy pipeline on a Colab T4 GPU.
The friend pipeline uses the same `gpu_fuzzy_trader` package but with
relaxed pool-admission thresholds, per-symbol Phase 2 training,
and the RB Governor engine for Phase 3+4.

Before running the cells:
- Connect the notebook to a Colab server (T4 GPU).
- Either upload the entire `friend_project/` folder to Colab, or let the bootstrap cell clone it from GitHub.
- If the folder lands somewhere else, set `PROJECT_ROOT` in the bootstrap cell to the uploaded path.
- If `train.csv` and `test.csv` live on Google Drive, the bootstrap cell copies them to `/content/friend_project_data/` (local disk) and links them into the repo `data/` folder so the pipeline does not read CSVs from Drive on the hot path.

The bootstrap cell mounts Google Drive when needed; the run cell writes outputs to `/content/friend_project_outputs/` and syncs to Drive after success.

**Important:** This is the *friend* project. All outputs go to `MyDrive/friend_project/outputs/`.


In [ ]:
import os
from getpass import getpass

github_token = os.environ.get("GITHUB_TOKEN", "").strip()
if not github_token:
    github_token = getpass("GitHub classic PAT (optional, press Enter to skip): ").strip()
if github_token:
    os.environ["GITHUB_TOKEN"] = github_token


In [ ]:
from base64 import b64encode
from pathlib import Path
from urllib.parse import urlsplit
import os
import shutil
import subprocess
import sys


PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "").strip() or None
DEFAULT_GITHUB_REPO_URL = "https://github.com/m-danaee/trading_platform.git"
GITHUB_REPO_URL = os.environ.get("GITHUB_REPO_URL", DEFAULT_GITHUB_REPO_URL).strip() or DEFAULT_GITHUB_REPO_URL
GITHUB_BRANCH = os.environ.get("GITHUB_BRANCH", "main").strip() or "main"
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN", "").strip() or None
GITHUB_CLONE_DIR = Path(os.environ.get("GITHUB_CLONE_DIR", "/content/friend_project")).expanduser()
os.environ["DATA_ROOT"] = "/content/drive/MyDrive"
os.environ["TRAIN_CSV_PATH"] = "/content/drive/MyDrive/train.csv"
os.environ["TEST_CSV_PATH"] = "/content/drive/MyDrive/test.csv"
DATA_ROOT = os.environ.get("DATA_ROOT", "").strip() or None
TRAIN_CSV_PATH = os.environ.get("TRAIN_CSV_PATH", "").strip() or None
TEST_CSV_PATH = os.environ.get("TEST_CSV_PATH", "").strip() or None


def _is_project_root(path: Path) -> bool:
    return (path / "gpu_fuzzy_trader").is_dir() and (path / "requirements.txt").is_file()


def _friend_package_dir(repo_root: Path) -> Path | None:
    """Return friend_project/ when the full monorepo was cloned."""
    nested = repo_root / "friend_project"
    if (nested / "gpu_fuzzy_trader").is_dir() and (nested / "requirements.txt").is_file():
        return nested
    return None


def _resolve_friend_package_root(path: Path) -> Path:
    """Prefer friend_project/gpu_fuzzy_trader over monorepo gpu_fuzzy_trader."""
    nested = _friend_package_dir(path)
    if nested is not None:
        return nested.resolve()
    return path.resolve()


def _discover_project_root() -> Path | None:
    if PROJECT_ROOT is not None:
        candidate = Path(PROJECT_ROOT).expanduser()
        if _is_project_root(candidate):
            return _resolve_friend_package_root(candidate)

    for candidate in [Path.cwd(), Path("/content/friend_project"), Path("/content")]:
        if _is_project_root(candidate):
            return _resolve_friend_package_root(candidate)

    content_root = Path("/content")
    if content_root.exists():
        for requirements_file in content_root.rglob("requirements.txt"):
            candidate = requirements_file.parent
            if _is_project_root(candidate):
                return _resolve_friend_package_root(candidate)

    return None


def _repo_url_has_credentials(repo_url: str) -> bool:
    parsed = urlsplit(repo_url)
    return bool(parsed.username or parsed.password)


def _ensure_drive_mounted() -> Path | None:
    drive_root = Path("/content/drive")
    if drive_root.exists():
        return drive_root

    try:
        from google.colab import drive
    except Exception:
        return None

    try:
        drive.mount("/content/drive", force_remount=False)
    except Exception:
        return None

    return drive_root if drive_root.exists() else None


def _path_has_csv_pair(root: Path) -> bool:
    return (root / "train.csv").is_file() and (root / "test.csv").is_file()


def _discover_dataset_paths(project_root: Path) -> tuple[Path, Path] | None:
    explicit_train = Path(TRAIN_CSV_PATH).expanduser() if TRAIN_CSV_PATH else None
    explicit_test = Path(TEST_CSV_PATH).expanduser() if TEST_CSV_PATH else None
    if explicit_train and explicit_test and explicit_train.is_file() and explicit_test.is_file():
        return explicit_train.resolve(), explicit_test.resolve()

    if DATA_ROOT:
        candidate = Path(DATA_ROOT).expanduser()
        if _path_has_csv_pair(candidate):
            return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

    for candidate in [
        project_root / "data",
        project_root,
        Path("/content/friend_project/data"),
        Path("/content/friend_project"),
    ]:
        if _path_has_csv_pair(candidate):
            return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

    drive_root = _ensure_drive_mounted()
    if drive_root is not None:
        drive_candidates = [
            drive_root / "MyDrive",
            drive_root / "My Drive",
            drive_root / "Shareddrives",
            drive_root / "Shared drives",
            drive_root,
        ]
        for candidate in drive_candidates:
            if _path_has_csv_pair(candidate):
                return (candidate / "train.csv").resolve(), (candidate / "test.csv").resolve()

        for candidate in drive_candidates:
            if not candidate.exists():
                continue
            for train_file in candidate.rglob("train.csv"):
                candidate_dir = train_file.parent
                if (candidate_dir / "test.csv").is_file():
                    return train_file.resolve(), (candidate_dir / "test.csv").resolve()

    content_root = Path("/content")
    if content_root.exists():
        for train_file in content_root.rglob("train.csv"):
            candidate = train_file.parent
            if (candidate / "test.csv").is_file():
                return train_file.resolve(), (candidate / "test.csv").resolve()

    return None


def _ensure_repo_data_files(project_root: Path, train_csv_path: Path, test_csv_path: Path) -> None:
    data_dir = project_root / "data"
    data_dir.mkdir(parents=True, exist_ok=True)

    targets = {
        data_dir / "train.csv": train_csv_path,
        data_dir / "test.csv": test_csv_path,
    }

    for target, source in targets.items():
        if target.exists():
            continue
        try:
            target.symlink_to(source)
        except Exception:
            shutil.copy2(source, target)


LOCAL_DATA_DIR = Path(os.environ.get("LOCAL_DATA_DIR", "/content/friend_project_data"))


def _stage_csv_to_local(source: Path, filename: str) -> Path:
    """Copy Drive-hosted CSVs to local Colab disk for faster pipeline I/O."""
    source = source.resolve()
    if not str(source).startswith("/content/drive"):
        return source
    LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
    target = (LOCAL_DATA_DIR / filename).resolve()
    if not target.is_file() or target.stat().st_size != source.stat().st_size:
        shutil.copy2(source, target)
    return target


def _repair_stale_numba_ops(project_root: Path) -> str:
    """Remove broken JIT empty branches left by older Colab notebook patches.
    The friend_project may not have numba_ops.py at all — handle gracefully."""
    numba_ops_path = project_root / "gpu_fuzzy_trader" / "evolution" / "numba_ops.py"
    if not numba_ops_path.is_file():
        return "numba_ops.py not present (friend_project)"

    text = numba_ops_path.read_text(encoding="utf-8")
    original = text
    for block in (
        "        if n == 0:\n            return [[]]\n\n",
        "        if n == 0:\n            return [[-1]]\n\n",
        "        if n == 0:\n            return [[]]\n",
        "        if n == 0:\n            return [[-1]]\n",
    ):
        text = text.replace(block, "")

    wrapper_guard = (
        "    if obj.shape[0] == 0:\n"
        "        return [[]]\n"
    )
    if wrapper_guard not in text and "def non_dominated_sort" in text:
        needle = "    obj = np.asarray(objectives, dtype=np.float64)\n"
        if needle in text:
            text = text.replace(
                needle,
                needle + wrapper_guard,
                1,
            )

    if text != original:
        numba_ops_path.write_text(text, encoding="utf-8")
        return "repaired stale JIT empty branch in numba_ops.py"

    jit_parts = text.split("def _non_dominated_sort_numba", 1)
    if len(jit_parts) > 1:
        jit_body = jit_parts[1].split("\n    @", 1)[0].split("\n    def ", 1)[0]
        if "if n == 0:" in jit_body:
            raise RuntimeError(
                "numba_ops.py still has a Numba-incompatible empty branch. "
                "Delete /content/friend_project and rerun bootstrap, or push the latest repo and git pull."
            )

    return "numba_ops.py ok"


def _try_git_pull(project_root: Path) -> str:
    git_dir = project_root / ".git"
    if not git_dir.is_dir():
        return "not a git repo"
    result = subprocess.run(
        ["git", "-C", str(project_root), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    if result.returncode == 0:
        return (result.stdout or "").strip() or "git pull ok"
    return f"git pull skipped: {(result.stderr or result.stdout or '').strip()}"


def _looks_like_auth_failure(output: str) -> bool:
    text = output.lower()
    return any(marker in text for marker in [
        'authentication failed',
        'could not read username',
        'repository not found',
        'terminal prompts disabled',
        'support for password authentication was removed',
        'access denied',
    ])


def _basic_auth_header(token: str) -> str:
    token_bytes = f'x-access-token:{token}'.encode('utf-8')
    encoded = b64encode(token_bytes).decode('ascii')
    return f'AUTHORIZATION: basic {encoded}'


def _run_git_clone(repo_url: str, target_dir: Path, token: str | None = None) -> subprocess.CompletedProcess[str]:
    clone_cmd = [
        'git',
        'clone',
        '--depth',
        '1',
        '--branch',
        GITHUB_BRANCH,
        repo_url,
        str(target_dir),
    ]
    clone_env = os.environ.copy()
    clone_env['GIT_TERMINAL_PROMPT'] = '0'
    clone_kwargs = {'capture_output': True, 'text': True, 'env': clone_env}
    if token:
        clone_cmd = [
            'git',
            '-c',
            f'http.extraheader={_basic_auth_header(token)}',
            'clone',
            '--depth',
            '1',
            '--branch',
            GITHUB_BRANCH,
            repo_url,
            str(target_dir),
        ]
    return subprocess.run(clone_cmd, **clone_kwargs)


def _clone_private_repo(repo_url: str, target_dir: Path) -> Path:

    if not repo_url:
        raise ValueError("GITHUB_REPO_URL is empty.")

    if target_dir.exists():
        if _is_project_root(target_dir):
            _try_git_pull(target_dir)
            return _resolve_friend_package_root(target_dir)
        if target_dir.is_dir() and any(target_dir.iterdir()):
            raise FileExistsError(
                f"{target_dir} already exists and is not the project root. "
                "Change GITHUB_CLONE_DIR or remove the conflicting directory."
            )
    target_dir.mkdir(parents=True, exist_ok=True)

    clone_result = _run_git_clone(repo_url, target_dir, GITHUB_TOKEN)
    if clone_result.returncode != 0:
        combined_output = (clone_result.stdout or '') + '\n' + (clone_result.stderr or '')
        hint = ""
        if _looks_like_auth_failure(combined_output):
            hint = (
                'This looks like a GitHub auth problem. For a classic PAT, use a token '
                'with repo read access, store it in Cell 1, and make sure the repository URL is correct.'
            )
        raise RuntimeError(
            'Git clone failed.\n'
            f'{combined_output.strip() or "(no git output)"}\n'
            f'{hint}'
        )

    return _resolve_friend_package_root(target_dir)


project_root = _discover_project_root()
if project_root is None and GITHUB_REPO_URL:
    project_root = _clone_private_repo(GITHUB_REPO_URL, GITHUB_CLONE_DIR)

if project_root is None:
    content_entries = []
    content_root = Path("/content")
    if content_root.exists():
        for path in sorted(content_root.iterdir()):
            content_entries.append(f"{path.name}/" if path.is_dir() else path.name)
    raise FileNotFoundError(
        "Could not find the project folder on the Colab server.\n"
        "Either upload the entire friend_project/ folder, or set GITHUB_REPO_URL "
        "(and store your classic PAT in Cell 1 if the repo is private) and rerun it.\n"
        f"/content currently contains: {', '.join(content_entries) if content_entries else '(empty or unavailable)'}"
    )

resolved_dataset_paths = _discover_dataset_paths(project_root)
if resolved_dataset_paths is None:
    raise FileNotFoundError(
        "Could not find train.csv and test.csv.\n"
        "Either set DATA_ROOT or TRAIN_CSV_PATH / TEST_CSV_PATH to the Google Drive location, "
        "or upload both files to the project data/ directory and rerun Cell 2."
    )

train_csv_path, test_csv_path = resolved_dataset_paths
train_csv_path = _stage_csv_to_local(train_csv_path, "train.csv")
test_csv_path = _stage_csv_to_local(test_csv_path, "test.csv")
_ensure_repo_data_files(project_root, train_csv_path, test_csv_path)
numba_ops_status = _repair_stale_numba_ops(project_root)
if (project_root / ".git").is_dir():
    git_pull_status = _try_git_pull(project_root)
    numba_ops_status = _repair_stale_numba_ops(project_root)
else:
    git_pull_status = "not a git repo"
os.environ["TRAIN_CSV_PATH"] = str(train_csv_path)
os.environ["TEST_CSV_PATH"] = str(test_csv_path)
if train_csv_path.parent == test_csv_path.parent:
    os.environ["DATA_ROOT"] = str(train_csv_path.parent)

PROJECT_ROOT = project_root
os.environ["PROJECT_ROOT"] = str(project_root)
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

existing_pythonpath = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = (
    str(project_root)
    if not existing_pythonpath
    else f"{project_root}{os.pathsep}{existing_pythonpath}"
)

print(f"Project root: {project_root}")
print(f"Python: {sys.executable}")
print(f"GitHub repo URL configured: {bool(GITHUB_REPO_URL)}")
print(f"GitHub URL has embedded credentials: {_repo_url_has_credentials(GITHUB_REPO_URL)}")
print(f"Train CSV: {train_csv_path}")
print(f"Test CSV: {test_csv_path}")
print(f"Numba ops: {numba_ops_status}")
print(f"Git sync: {git_pull_status}")
print(f"Repo data links ready: {(project_root / 'data' / 'train.csv').exists() and (project_root / 'data' / 'test.csv').exists()}")
print(f"Upload detected under /content: {str(project_root).startswith('/content')}")


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PIP = [sys.executable, "-m", "pip", "install", "-q"]

_in_colab = (
    os.environ.get("COLAB_RELEASE_TAG") is not None
    or Path("/content").is_dir()
    or importlib.util.find_spec("google.colab") is not None
)

# Pin set aligned with requirements.txt and tested in local .venv.
# On Colab T4 (CUDA 12.x), install JAX via jax[cuda12] *after* core deps so
# generic CPU jaxlib wheels from requirements.txt do not win.
COLAB_CORE_PINS = [
    "numpy>=1.26,<2.4",
    "pandas>=2.0,<3",
    "numba>=0.58,<0.66",
    "scikit-learn>=1.3,<2",
    "matplotlib>=3.7,<4",
    "pyarrow>=14.0",
    "optuna>=3.5.0,<5",
]
COLAB_EVOX = "evox>=1.0.0,<2"
COLAB_JAX = "jax[cuda12]==0.10.1"
# EvoX NSGA-III selection uses PyTorch tensors; CPU wheel is enough and smaller.
COLAB_TORCH_CPU = (
    "torch==2.5.1",
    "--index-url",
    "https://download.pytorch.org/whl/cpu",
)

if _in_colab:
    print("Colab T4: installing pinned GPU stack (CUDA 12 + JAX 0.10.1 + EvoX)...")
    subprocess.check_call(PIP + ["-U", "pip", "wheel"])
    subprocess.check_call(PIP + COLAB_CORE_PINS)
    subprocess.check_call(PIP + list(COLAB_TORCH_CPU))
    subprocess.check_call(PIP + [COLAB_EVOX])
    # Install GPU JAX last to override Colab/previous CPU jaxlib wheels.
    subprocess.check_call(PIP + ["-U", COLAB_JAX])

    import jax

    print(
        f"JAX {jax.__version__} | backend={jax.default_backend()} | "
        f"devices={jax.devices()}"
    )
    if jax.default_backend() != "gpu":
        raise RuntimeError(
            "JAX is not using the GPU. Choose Runtime > Change runtime type > "
            "T4 GPU, then Runtime > Restart session and rerun from Cell 1."
        )
else:
    print("Installing dependencies from requirements.txt ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"])
    # Local NVIDIA GPU (e.g. RTX 4050 on WSL): install CUDA plugin after CPU wheels.
    # Use default PyPI; plugin versions must match jaxlib (0.10.1).
    LOCAL_GPU_PINS = [
        "jax-cuda13-plugin==0.10.1",
        "jax-cuda13-pjrt==0.10.1",
    ]
    try:
        subprocess.check_call(PIP + ["-U"] + LOCAL_GPU_PINS)
        import jax
        print(
            f"JAX {jax.__version__} | backend={jax.default_backend()} | "
            f"devices={jax.devices()}"
        )
    except Exception as exc:
        print(f"GPU JAX install skipped ({exc}); Phase 2 may use CPU backtests.")


In [ ]:
import importlib.metadata as _im
import os
import jax
from pathlib import Path

# RAM optimisation: lower batch size and scan unroll to reduce JAX compile
# memory spike on Colab's 12 GiB host RAM.
os.environ["PHASE2_GPU_BATCH_SIZE"] = "64"
os.environ["PHASE2_GPU_BATCH_SIZE_AUTO"] = "false"
# XLA memory fraction: leave headroom for JAX compile on constrained hosts.
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.70")

from gpu_fuzzy_trader._jax_env import configure_jax_env

configure_jax_env()

from gpu_fuzzy_trader.run_pipeline import Pipeline_Orchestrator
from gpu_fuzzy_trader import config as cfg

# Monkeypatch scan unroll: halves XLA compile RAM without changing config.py.
# Fewer kernel launches means a small speed trade-off but avoids OOM SIGKILL.
cfg.PHASE2_SCAN_UNROLL = 16

# Per-symbol Phase 2: run Phase 2 separately for each symbol.
cfg.PER_SYMBOL_PHASE2 = True

# Two-stage Phase 2: Stage A exploration then Stage B refinement (32+18 gens).
cfg.PHASE2_TWO_STAGE_ENABLED = False

# Phase 5: remove negative-PnL rules from final strategy (OOS evaluator).
cfg.PHASE5_REMOVE_NEGATIVE_PNL_RULES = True

print("Import OK")
print(f"TRAIN_CSV_PATH: {Path(cfg.TRAIN_CSV_PATH).resolve()}")
print(f"TEST_CSV_PATH: {Path(cfg.TEST_CSV_PATH).resolve()}")
print(f"JAX backend: {jax.default_backend()} | devices: {jax.devices()}")
for _pkg in ("evox", "optuna", "numba", "torch", "jax"):
    try:
        print(f"{_pkg}: {_im.version(_pkg)}")
    except _im.PackageNotFoundError:
        print(f"{_pkg}: not installed")

try:
    import torch
    from evox.operators.sampling.uniform import uniform_sampling
    from evox.operators.selection.non_dominate import non_dominate_rank

    print("EvoX NSGA-III ready")
except ImportError as _evox_exc:
    raise RuntimeError(
        "EvoX operators failed to import — Phase 2 will fall back to NSGA-II. "
        f"Re-run the install cell or upgrade evox/torch. ({_evox_exc})"
    ) from _evox_exc

from gpu_fuzzy_trader._gpu_runtime import (
    detect_gpu_vram_gb,
    resolve_phase2_gpu_batch_size,
)

_vram = detect_gpu_vram_gb()
_vram_s = f"{_vram:.1f} GiB" if _vram is not None else "unknown"
_jax_cache = os.environ.get("JAX_COMPILATION_CACHE_DIR", "(not set)")
print(
    f"Colab GPU defaults: enabled={cfg.is_colab_runtime()} | "
    f"phase3_gpu={cfg.PHASE3_USE_GPU} | "
    f"batch_auto={cfg.PHASE2_GPU_BATCH_SIZE_AUTO} | "
    f"jax_cache={_jax_cache}"
)
print(
    f"PHASE2_USE_GPU={cfg.PHASE2_USE_GPU} | "
    f"batch_size={resolve_phase2_gpu_batch_size()} "
    f"(config={cfg.PHASE2_GPU_BATCH_SIZE}) | "
    f"scan_unroll={cfg.PHASE2_SCAN_UNROLL} | vram={_vram_s}"
)
print(
    f"Friend package: {Path(cfg.__file__).resolve().parent.parent} | "
    f"PER_SYMBOL_PHASE2={cfg.PER_SYMBOL_PHASE2} | "
    f"PHASE2_TWO_STAGE_ENABLED={cfg.PHASE2_TWO_STAGE_ENABLED} | "
    f"PHASE5_REMOVE_NEGATIVE_PNL_RULES={cfg.PHASE5_REMOVE_NEGATIVE_PNL_RULES}"
)
if not getattr(cfg, "PER_SYMBOL_PHASE2", False):
    raise RuntimeError(
        "Wrong gpu_fuzzy_trader package loaded (PER_SYMBOL_PHASE2 is False). "
        "Re-run the bootstrap cell so PROJECT_ROOT points at friend_project/."
    )
if jax.default_backend() != "gpu" and cfg.PHASE2_USE_GPU:
    print(
        "WARNING: PHASE2_USE_GPU=True but JAX backend is not GPU — "
        "Phase 2 will fall back to CPU backtests (much slower)."
    )

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import jax

# RAM optimisation: lower batch size to reduce JAX compile memory spike.
os.environ["PHASE2_GPU_BATCH_SIZE"] = "64"
os.environ["PHASE2_GPU_BATCH_SIZE_AUTO"] = "True"
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.70")

# Run controls:
#   RUN_PHASE : None = full pipeline, or 1..5 for a single phase.
#               With RB_GOVERNOR_ENABLED=True (config.py default),
#               --phase 3 and --phase 4 both dispatch to the RB Governor
#               (combined Phase 3 + Phase 4 replacement).
#   RESUME    : skip already-completed outputs.
RUN_PHASE = None
RESUME = False
USE_LOCAL_SCRATCH = True
LOCAL_OUTPUT_DIR = Path("/content/friend_project_outputs")
DRIVE_OUTPUT_DIR = Path("/content/drive/MyDrive/friend_project/outputs")
OUTPUT_DIR = str(LOCAL_OUTPUT_DIR if USE_LOCAL_SCRATCH else DRIVE_OUTPUT_DIR)
PHASE2_ARCHIVE_SRC = Path(PROJECT_ROOT) / "phase2_rule_archive"
PHASE2_ARCHIVE_DST = Path("/content/drive/MyDrive/friend_project/phase2_rule_archive")
POOL_DIR_SRC = Path(PROJECT_ROOT) / "pools" / "per_symbol"
POOL_DIR_DST = Path("/content/drive/MyDrive/friend_project/pools/per_symbol")

print(f"JAX backend: {jax.default_backend()} | devices: {jax.devices()}", flush=True)
print(f"Pipeline cwd (friend package): {PROJECT_ROOT}", flush=True)

if USE_LOCAL_SCRATCH:
    LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if RESUME and DRIVE_OUTPUT_DIR.is_dir():
        shutil.copytree(DRIVE_OUTPUT_DIR, LOCAL_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"Resume: copied prior outputs from Drive -> {LOCAL_OUTPUT_DIR}", flush=True)

pipeline_cmd = [
    sys.executable,
    "-u",
    "-m",
    "gpu_fuzzy_trader.run_pipeline",
    "--output",
    OUTPUT_DIR,
]
if RUN_PHASE is not None:
    pipeline_cmd.extend(["--phase", str(RUN_PHASE)])
if RESUME:
    pipeline_cmd.append("--resume")

print("Running:", " ".join(pipeline_cmd), flush=True)
t0 = time.perf_counter()

process = subprocess.Popen(
    pipeline_cmd,
    cwd=str(PROJECT_ROOT),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env={**os.environ, "PYTHONUNBUFFERED": "1"},
)

assert process.stdout is not None
for line in iter(process.stdout.readline, ""):
    print(line, end="", flush=True)

return_code = process.wait()
elapsed = time.perf_counter() - t0
if return_code != 0:
    raise subprocess.CalledProcessError(return_code, pipeline_cmd)

print(f"\nPipeline finished in {elapsed:.1f}s", flush=True)

if USE_LOCAL_SCRATCH:
    _ensure_drive_mounted()
    if DRIVE_OUTPUT_DIR.parent.exists():
        DRIVE_OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(LOCAL_OUTPUT_DIR, DRIVE_OUTPUT_DIR, dirs_exist_ok=True)
        print(f"Outputs synced to Google Drive: {DRIVE_OUTPUT_DIR}", flush=True)

if PHASE2_ARCHIVE_SRC.is_dir():
    PHASE2_ARCHIVE_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(PHASE2_ARCHIVE_SRC, PHASE2_ARCHIVE_DST, dirs_exist_ok=True)
    print(f"Phase 2 archive synced to Google Drive: {PHASE2_ARCHIVE_DST}", flush=True)
else:
    print(f"Phase 2 archive folder not found at: {PHASE2_ARCHIVE_SRC}", flush=True)

if POOL_DIR_SRC.is_dir():
    POOL_DIR_DST.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(POOL_DIR_SRC, POOL_DIR_DST, dirs_exist_ok=True)
    print(f"Per-symbol pools synced to Google Drive: {POOL_DIR_DST}", flush=True)

print("\nPipeline finished successfully.", flush=True)

After the pipeline finishes, artifacts are written to local `/content/friend_project_outputs/` during the run, then synced to `MyDrive/friend_project/outputs/` on Google Drive.

**Per-symbol pools** are also synced to `MyDrive/friend_project/pools/per_symbol/`.

**Phase 2 archive** is synced to `MyDrive/friend_project/phase2_rule_archive/`.

With `RB_GOVERNOR_ENABLED = True` in `config.py` (the default), Phases 3 + 4 are handled by the RB Governor pipeline (integrated rule selection, risk-grid search, and profit amplifier) and write `outputs/long.json` and `outputs/short.json` in the same schema as before. To restore the legacy Phase 3 + Phase 4 modules, set `RB_GOVERNOR_ENABLED = False`.

For partial reruns set `RESUME = True` and optionally `RUN_PHASE` to `1`–`5` (`3` and `4` both dispatch to the governor). Open the friend's `evaluator_v5` notebook and point it at the generated `outputs/long.json` and `outputs/short.json` files.

**Key friend settings activated in this notebook:**
- `PER_SYMBOL_PHASE2 = True` — runs Phase 2 separately for each symbol
- `PHASE2_TWO_STAGE_ENABLED = False` — set `True` for Stage A (32 gen) → Stage B (18 gen) two-stage search
- `PHASE5_REMOVE_NEGATIVE_PNL_RULES = True` — prunes negative-PnL rules from final strategy
- `PHASE2_SCAN_UNROLL = 16` — reduced for Colab RAM
- Relaxed pool-admission thresholds (from `config.py` defaults)
